# 20 V-web: velocity-shear labels on TNG300-3-Dark

The V-web (Hoffman et al. 2012, [MNRAS 425, 2049](https://arxiv.org/abs/1201.3367)) classifies each point by the shear of the peculiar-velocity field instead of the tidal tensor:

$$\Sigma_{ij} = -\frac{1}{2H_0}\left(\frac{\partial v_i}{\partial r_j} + \frac{\partial v_j}{\partial r_i}\right),$$

with H₀ = 100 h km/s/Mpc. Positions are in h⁻¹ Mpc and velocities in km/s, so in the code H₀ is the number 100 and Σ is dimensionless; the factors of h cancel.

With the minus sign a positive eigenvalue means compression along that axis. Class = number of eigenvalues above a threshold μ_th: 0 void, 1 wall (sheet), 2 filament, 3 halo (knot), the same counting as our T-web labels.

**There is no standard threshold.** Hoffman et al. picked 0.44 by eye for their resolution (0.25 h⁻¹ Mpc smoothing, 64 h⁻¹ Mpc box) and say the value has to be set for each problem. So this notebook is for looking: slices at several thresholds, class fractions and percolation against the threshold, and a per-voxel comparison with our T-web labels.

**Reference threshold from linear theory.** With ∇²φ = δ and the linear velocity v = −f H₀ ∇φ, the shear is Σ = f T. A V-web threshold of f λ_th then gives the same classes as the T-web at λ_th. For TNG (Ω_m = 0.3089, f ≈ Ω_m^0.55 ≈ 0.52) and λ_th = 0.65 this is ≈ 0.34. It is a reference line only; where the V-web departs from f × T-web is where the dynamics are nonlinear (section 5).

**How the velocity field is made.** Dark-matter particles of TNG300-3-Dark snapshot 99 (the same run and snapshot as our density and T-web mask), velocities converted from TNG's km √a / s to peculiar km/s (× √a, a = 1 here). Each velocity component and the particle count are put on the 512³ grid with CIC (number weighting, as Hoffman et al.), both are smoothed with the same Gaussian as the T-web mask (σ = 2.4 cells, σ_eff ≈ 1.04 h⁻¹ Mpc), and the smoothed momentum is divided by the smoothed count. Hoffman et al. instead divide in each cell and then smooth; they had ~64 particles per cell, we have ~1.8, so many void cells hold no particle. Section 4 shows how sparse the counts get in voids.

**Data.** The particles come from the TNG API, one snapshot chunk at a time, only the fields listed in `FIELDS` (positions and velocities; particle IDs too, for shell-crossing labels later). Downloaded once to `data/TNG/TNG300-3-Dark_snap99_dm/` and read from there afterwards. If the API fails, the TNG Lab has the same files (`sims.TNG/TNG300-3-Dark/output/snapdir_099/`): run sections 1 and 3 there with `SNAP_DIR` pointing at that folder and download the velocity file it writes.

Outputs go to `deepvoid_runs/vweb/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

## 1 Settings

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.patches import Patch
from scipy import ndimage

from deepvoid import Catalog
from deepvoid.evaluate import confusion_matrix
from deepvoid.grid import velocity_field
from deepvoid.io import load_volume, save_volume
from deepvoid.labels import CLASS_NAMES, tweb_eigenvalues, vweb_eigenvalues
from deepvoid.sources.tng import fetch_snapshot_dm, read_snapshot_dm, snapshot_files
from deepvoid.voids import percolation
from deepvoid.volume import Volume

RUNS = Path('/content/drive/MyDrive/deepvoid_runs')
OUT = RUNS / 'vweb'
OUT.mkdir(parents=True, exist_ok=True)
catalog = Catalog()

SIM, SNAPSHOT = 'TNG300-3-Dark', 99
SNAP_DIR = catalog.root / 'TNG' / 'TNG300-3-Dark_snap99_dm'   # chunk files (API download, or the Lab's snapdir_099)
FIELDS = ('Coordinates', 'Velocities', 'ParticleIDs')          # IDs are not used here; kept for shell-crossing labels
N = 512
SIGMA_CELLS = 2.4              # Gaussian sigma in cells, as the TNG300-3 T-web mask (sigma_eff ~ 1.04 h^-1 Mpc)
LAMBDA_TH = 0.65               # T-web threshold of our labels
OMEGA_M = 0.3089               # TNG (Planck 2015)
F_GROWTH = OMEGA_M ** 0.55     # linear growth rate f, approximation
LINEAR_MATCH = F_GROWTH * LAMBDA_TH
HOFFMAN = 0.44                 # Hoffman et al. 2012 (their resolution, chosen by eye)
SLICE_THRESHOLDS = [0.1, 0.2, round(LINEAR_MATCH, 2), HOFFMAN]   # V-web thresholds drawn in section 6
SCAN = np.round(np.arange(0.0, 1.0001, 0.05), 2)                 # class fractions, section 7
PERC_SCAN = np.round(np.arange(0.0, 0.8001, 0.1), 2)             # percolation, section 8 (slow)
CHOSEN = round(LINEAR_MATCH, 2)   # V-web threshold for section 9; change after looking at the slices
SAVE_LABELS = False               # section 10: write V-web labels at CHOSEN
SLICE = None                      # x index of the slices (None: middle)
ZOOM = 50.0                       # h^-1 Mpc, side of the zoomed slices
SEED = 0

COLORS = ['white', '#b0b0b0', '#555555', 'black']   # void, wall, filament, halo (as Hoffman et al.)
CMAP = ListedColormap(COLORS)
summary = {'sigma_cells': SIGMA_CELLS, 'linear_match': LINEAR_MATCH, 'f_growth': F_GROWTH}
print(f'f = {F_GROWTH:.3f}; V-web threshold matching the T-web at {LAMBDA_TH} in linear theory: {LINEAR_MATCH:.3f}')

## 2 Particles

Downloads the chunks not yet on Drive (TNG300-3-Dark has 4). Needs the API key from your tng-project.org profile as the Colab secret `TNG_API_KEY`. Each chunk is checked after download (all requested fields present and complete); a failed or cut-off download is retried every minute, up to 15 times. Download time depends on the TNG server: chunk sizes are printed as they finish.

In [ ]:
def have_all_chunks():
    try:
        return len(snapshot_files(SNAP_DIR)) >= 4
    except FileNotFoundError:
        return False


if have_all_chunks():
    print('chunks on Drive:', [f.name for f in snapshot_files(SNAP_DIR)])
else:
    from google.colab import userdata
    t0 = time.time()
    files = fetch_snapshot_dm(SNAP_DIR, userdata.get('TNG_API_KEY'), sim=SIM, snapshot=SNAPSHOT, fields=FIELDS)
    print(f'{len(files)} chunks in {(time.time() - t0) / 60:.1f} min')

## 3 Velocity field

One chunk at a time (each ~61 million particles, ~1.5 GB as float32 positions and velocities), so the whole snapshot is never in memory. CIC for the count and the three momentum components, then smoothing and division. Saved to `vweb/velocity_n512_s2.4.npz` and read from there on reruns. Expect roughly 10–20 min for the gridding (four CIC passes over 244 million particles); progress is printed per chunk.

In [ ]:
VEL_FILE = OUT / f'velocity_n{N}_s{SIGMA_CELLS}.npz'
COUNT_FILE = OUT / f'count_n{N}_s{SIGMA_CELLS}.npz'


def chunks():
    files = snapshot_files(SNAP_DIR)
    t0 = time.time()
    for i, f in enumerate(files):
        t = read_snapshot_dm(f, sim=SIM)
        print(f'chunk {i + 1}/{len(files)}: {len(t):,} particles, read at {time.time() - t0:.0f} s')
        yield t


if VEL_FILE.exists() and COUNT_FILE.exists():
    vel_data = np.load(VEL_FILE)
    velocity = [Volume(vel_data[c], float(vel_data['box_size']), {'component': c}) for c in 'xyz']
    vel_meta = json.loads(str(vel_data['meta']))
    count = load_volume(COUNT_FILE)
else:
    t0 = time.time()
    velocity, count = velocity_field(chunks(), N, sigma_cells=SIGMA_CELLS, return_count=True)
    vel_meta = {k: v for k, v in velocity[0].meta.items() if k != 'component'}
    np.savez(VEL_FILE, x=velocity[0].data, y=velocity[1].data, z=velocity[2].data,
             box_size=velocity[0].box_size, meta=json.dumps(vel_meta))
    save_volume(count, COUNT_FILE)
    print(f'velocity field in {(time.time() - t0) / 60:.1f} min')
box = velocity[0].box_size
n = velocity[0].n
cell = box / n
SLICE = n // 2 if SLICE is None else SLICE
summary['velocity_meta'] = vel_meta
print(vel_meta)
print('rms velocity per component (km/s):', [round(float(v.data.std()), 1) for v in velocity])

## 4 Checks: grid alignment, T-web, sparse cells

- **Alignment.** The smoothed particle count should match the catalog density smoothed the same way, if both use the same grid points and axis order. The correlation of their logarithms is printed, with the axes swapped (x ↔ z) as a control.
- **T-web.** The T-web recomputed from the catalog density at σ = 2.4 cells and λ_th = 0.65 should reproduce the catalog mask (notebook 00 found 100.00%).
- **Sparse cells.** The smoothed count per cell (mean 625³ / 512³ ≈ 1.82) in each T-web class: how few particles set the velocity in voids.

In [ ]:
dm = catalog.load('tng300-3/dm')
mask = catalog.load('tng300-3/mask').data.astype(np.uint8)
rho_s = ndimage.gaussian_filter(dm.data / dm.data.mean(), SIGMA_CELLS, mode='wrap')
c = np.log(count.data / count.data.mean() + 1e-3).ravel()
r = np.log(rho_s + 1e-3)
corr = float(np.corrcoef(c, r.ravel())[0, 1])
corr_swapped = float(np.corrcoef(c, np.swapaxes(r, 0, 2).ravel())[0, 1])
del c, r
print(f'log smoothed count vs log smoothed density: r = {corr:.4f} (x and z swapped: {corr_swapped:.4f})')
summary['alignment_corr'] = corr
summary['alignment_corr_swapped'] = corr_swapped

eig_t = tweb_eigenvalues(dm.data, SIGMA_CELLS)
tweb = (eig_t > LAMBDA_TH).sum(-1).astype(np.uint8)
agree = float((tweb == mask).mean())
print(f'recomputed T-web vs catalog mask: {100 * agree:.2f}% of voxels agree')
summary['tweb_agreement'] = agree

print(f'cells with no smoothed count (velocity set to 0): {vel_meta["empty_cells"]}')
fig, ax = plt.subplots(figsize=(7, 4.5))
bins = np.logspace(-3, 3, 121)
for k, name in enumerate(CLASS_NAMES):
    if not (mask == k).any():
        continue
    ax.hist(count.data[mask == k], bins=bins, histtype='step', color=['tab:blue', 'tab:orange', 'tab:green', 'tab:red'][k],
            label=f'T-web {name}', density=True)
ax.axvline(count.data.mean(), color='k', ls='--', label=f'mean ({count.data.mean():.2f})')
ax.set_xscale('log')
ax.set_xlabel('smoothed particle count per cell')
ax.set_ylabel('probability density')
ax.set_title(f'Smoothed particle count per cell by T-web class (σ = {SIGMA_CELLS} cells)')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.savefig(OUT / 'count_by_class.png', dpi=120, bbox_inches='tight')
plt.show()
for q in (0.01, 0.1, 0.5):
    print(f'T-web voids: {100 * q:.0f}% of cells have a smoothed count below {np.quantile(count.data[mask == 0], q):.3f}')

## 5 Eigenvalues, and the V-web against f × T-web

V-web eigenvalues from the velocity field (cached in `vweb/eig_n512_s2.4.npy`, 1.6 GB). In linear theory each V-web eigenvalue equals f times the matching T-web eigenvalue; the 2D histograms show where they agree (on the line) and where the dynamics are nonlinear. A random 2 million voxels.

In [ ]:
EIG_FILE = OUT / f'eig_n{N}_s{SIGMA_CELLS}.npy'
if EIG_FILE.exists():
    eig_v = np.load(EIG_FILE)
else:
    t0 = time.time()
    eig_v = vweb_eigenvalues([v.data for v in velocity], box)
    np.save(EIG_FILE, eig_v)
    print(f'V-web eigenvalues in {time.time() - t0:.0f} s')

rng = np.random.default_rng(SEED)
idx = rng.choice(n ** 3, size=2_000_000, replace=False)
ev, et = eig_v.reshape(-1, 3)[idx], F_GROWTH * eig_t.reshape(-1, 3)[idx]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
lim = tuple(np.quantile(np.concatenate([ev, et]), [0.0005, 0.9995]))
names = ['smallest', 'middle', 'largest']
summary['eig_corr'] = {}
for k, ax in enumerate(axes):
    h = ax.hist2d(et[:, k], ev[:, k], bins=200, range=[lim, lim], norm=LogNorm(), cmap='viridis')
    ax.plot(lim, lim, 'r--', lw=1, label='V-web = f × T-web (linear theory)')
    r = float(np.corrcoef(et[:, k], ev[:, k])[0, 1])
    summary['eig_corr'][names[k]] = r
    ax.set_title(f'{names[k]} eigenvalue (r = {r:.3f})')
    ax.set_xlabel(f'f × T-web eigenvalue (f = {F_GROWTH:.3f})')
    ax.set_ylabel('V-web eigenvalue')
fig.colorbar(h[3], ax=axes, label='voxels per bin', shrink=0.85)
fig.legend(*axes[0].get_legend_handles_labels(), loc='lower center', bbox_to_anchor=(0.45, -0.06))
fig.suptitle('V-web vs linear-theory prediction from the T-web, per voxel')
plt.savefig(OUT / 'eig_vs_tweb.png', dpi=120, bbox_inches='tight')
plt.show()
del ev, et

## 6 Slices

The same x-plane in every panel: log density, the T-web labels (λ_th = 0.65), and the V-web at each threshold in `SLICE_THRESHOLDS` (0.34 is the linear-theory match, 0.44 Hoffman et al.'s value). Then the same panels zoomed into a `ZOOM` h⁻¹ Mpc square.

In [ ]:
def show_slices(lo, hi, title, fname):
    s = np.s_[lo[0]:hi[0], lo[1]:hi[1]]
    ext = [lo[0] * cell, hi[0] * cell, lo[1] * cell, hi[1] * cell]
    panels = [('log10 ρ/ρ̄ (smoothed)', np.log10(np.maximum(rho_s[SLICE][s], 1e-2)), 'density'),
              (f'T-web, λ_th = {LAMBDA_TH}', tweb[SLICE][s], 'labels')]
    panels += [(f'V-web, μ_th = {t}', (eig_v[SLICE] > t).sum(-1)[s], 'labels') for t in SLICE_THRESHOLDS]
    cols = 3
    rows = int(np.ceil(len(panels) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5.2 * cols, 5 * rows), squeeze=False)
    for ax, (name, img, kind) in zip(axes.ravel(), panels):
        if kind == 'density':
            im = ax.imshow(img.T, origin='lower', extent=ext, cmap='magma', vmin=-1, vmax=2)
            fig.colorbar(im, ax=ax, label='log10 ρ/ρ̄', shrink=0.8)
        else:
            ax.imshow(img.T, origin='lower', extent=ext, cmap=CMAP, vmin=-0.5, vmax=3.5, interpolation='nearest')
        ax.set_title(name)
        ax.set_xlabel('y (h⁻¹ Mpc)')
        ax.set_ylabel('z (h⁻¹ Mpc)')
    for ax in axes.ravel()[len(panels):]:
        ax.axis('off')
    handles = [Patch(facecolor=c, edgecolor='k', label=l) for c, l in zip(COLORS, CLASS_NAMES)]
    fig.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.0, 0.95), title='class')
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(OUT / fname, dpi=120, bbox_inches='tight')
    plt.show()


show_slices((0, 0), (n, n), f'T-web and V-web at x = {SLICE * cell:.1f} h⁻¹ Mpc (one cell thick)', 'slices.png')
z = int(ZOOM / cell)
c0 = (n - z) // 2
show_slices((c0, c0), (c0 + z, c0 + z), f'Zoom: {ZOOM:.0f} h⁻¹ Mpc square at x = {SLICE * cell:.1f} h⁻¹ Mpc', 'slices_zoom.png')

## 7 Class fractions against the threshold

Volume share of each V-web class against μ_th (solid), with the T-web shares at λ_th = 0.65 as horizontal dashed lines in the same colors. Vertical lines: the linear-theory match and Hoffman et al.'s 0.44.

In [ ]:
frac_v = np.array([np.bincount((eig_v > t).sum(-1).ravel(), minlength=4) / n ** 3 for t in SCAN])
frac_t = np.bincount(tweb.ravel(), minlength=4) / n ** 3
summary['fractions'] = {'scan': SCAN.tolist(), 'vweb': frac_v.tolist(), 'tweb': frac_t.tolist()}
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
for k, name in enumerate(CLASS_NAMES):
    ax.plot(SCAN, frac_v[:, k], '-o', ms=3, color=colors[k], label=f'V-web {name}')
    ax.axhline(frac_t[k], color=colors[k], ls='--', lw=1, label=f'T-web {name} (λ_th = {LAMBDA_TH})')
ax.axvline(LINEAR_MATCH, color='k', ls=':', label=f'linear-theory match ({LINEAR_MATCH:.2f})')
ax.axvline(HOFFMAN, color='gray', ls='-.', label=f'Hoffman et al. 2012 ({HOFFMAN})')
ax.set_yscale('log')
ax.set_ylim(1e-4, 1.1)
ax.set_xlabel('V-web threshold μ_th')
ax.set_ylabel('volume fraction')
ax.set_title('V-web class volume fractions against the threshold')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8)
plt.tight_layout()
plt.savefig(OUT / 'fractions.png', dpi=120, bbox_inches='tight')
plt.show()

## 8 Does the void class percolate?

Share of the void volume in the largest periodic connected region (face neighbors), against the threshold: V-web on the left, T-web on the right for reference. Near 1 means one region fills almost all of the void class (it percolates); doc 15 found the T-web voids stop percolating between λ_th = 0.2 and 0. Hoffman et al. describe V-web voids "bisected by sheets"; this tests whether that holds at our smoothing. Each threshold takes about as long as one connected-component labeling of the 512³ box; elapsed time is printed.

In [ ]:
t0 = time.time()
perc_v, perc_t = [], []
for t in PERC_SCAN:
    perc_v.append(percolation((eig_v <= t).all(-1)))   # void: no eigenvalue above t, as the labels
    perc_t.append(percolation((eig_t <= t).all(-1)))
    print(f'threshold {t:.2f}: V-web largest share {perc_v[-1]["largest_share"]:.3f} '
          f'({perc_v[-1]["components"]} regions), T-web {perc_t[-1]["largest_share"]:.3f} '
          f'({perc_t[-1]["components"]} regions); {time.time() - t0:.0f} s')
summary['percolation'] = {'scan': PERC_SCAN.tolist(), 'vweb': perc_v, 'tweb': perc_t}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, perc, name, ref in ((axes[0], perc_v, 'V-web', LINEAR_MATCH), (axes[1], perc_t, 'T-web', LAMBDA_TH)):
    ax.plot(PERC_SCAN, [p['largest_share'] for p in perc], '-o', color='tab:blue', label='largest region / all void voxels')
    ax.axvline(ref, color='k', ls=':', label=f'{"linear-theory match" if name == "V-web" else "our labels"} ({ref:.2f})')
    ax.set_xlabel(f'{name} threshold')
    ax.set_title(f'{name}: share of the void volume in the largest region')
    ax.set_ylim(0, 1.05)
axes[0].set_ylabel('share of void volume')
axes[1].legend(loc='upper left', bbox_to_anchor=(1.02, 1))
axes[0].legend(loc='lower right')
plt.tight_layout()
plt.savefig(OUT / 'percolation.png', dpi=120, bbox_inches='tight')
plt.show()

## 9 V-web against T-web, per voxel

Confusion matrix of the V-web at `CHOSEN` against our T-web labels: each row is one T-web class, split by the V-web class its voxels get (rows sum to 1).

In [ ]:
vweb = (eig_v > CHOSEN).sum(-1).astype(np.uint8)
cm = confusion_matrix(tweb, vweb)
cm_row = cm / np.maximum(cm.sum(1, keepdims=True), 1)   # a class with no voxels shows as zeros
summary['confusion'] = {'threshold': CHOSEN, 'counts': cm.tolist()}
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm_row, cmap='Blues', vmin=0, vmax=1)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{cm_row[i, j]:.2f}', ha='center', va='center', color='white' if cm_row[i, j] > 0.5 else 'black')
ax.set_xticks(range(4), CLASS_NAMES)
ax.set_yticks(range(4), CLASS_NAMES)
ax.set_xlabel(f'V-web class (μ_th = {CHOSEN})')
ax.set_ylabel(f'T-web class (λ_th = {LAMBDA_TH})')
ax.set_title('Where each T-web class falls in the V-web (rows sum to 1)')
fig.colorbar(im, ax=ax, label='share of the T-web class')
plt.tight_layout()
plt.savefig(OUT / f'confusion_th{CHOSEN}.png', dpi=120, bbox_inches='tight')
plt.show()

## 10 Save

In [ ]:
(OUT / 'summary.json').write_text(json.dumps(summary, indent=1, default=float))
if SAVE_LABELS:
    meta = {**{k: v for k, v in vel_meta.items() if k not in ('field', 'units', 'empty_cells')},
            'field': 'mask', 'labels': 'vweb', 'threshold': CHOSEN}
    path = OUT / f'vweb_mask_n{N}_s{SIGMA_CELLS}_th{CHOSEN}.npz'
    save_volume(Volume(vweb, box, meta), path)
    print('saved', path)
print('summary written to', OUT / 'summary.json')